In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/notebooks/anushkadthakur15/01-sudoku-data/val.npz
/kaggle/input/notebooks/anushkadthakur15/01-sudoku-data/pool.npz
/kaggle/input/notebooks/anushkadthakur15/01-sudoku-data/__results__.html
/kaggle/input/notebooks/anushkadthakur15/01-sudoku-data/test.npz
/kaggle/input/notebooks/anushkadthakur15/01-sudoku-data/__notebook__.ipynb
/kaggle/input/notebooks/anushkadthakur15/01-sudoku-data/__output__.json
/kaggle/input/notebooks/anushkadthakur15/01-sudoku-data/custom.css


In [2]:
import glob
import numpy as np

files = glob.glob("/kaggle/input/**/*.npz", recursive=True)
print(files)

['/kaggle/input/notebooks/anushkadthakur15/01-sudoku-data/val.npz', '/kaggle/input/notebooks/anushkadthakur15/01-sudoku-data/pool.npz', '/kaggle/input/notebooks/anushkadthakur15/01-sudoku-data/test.npz']


In [3]:
# Why: we only augment training data, never the test set
for f in files:
    if f.endswith("pool.npz"):
        pool_file = f

data = np.load(pool_file)
puzzles = data["puzzles"]
solutions = data["solutions"]
groups = data["groups"]

print(puzzles.shape)

(98000, 81)


In [4]:
# Why: to look at puzzles with our own eyes
def show_grid(cells):
    grid = cells.reshape(9, 9)
    for r in range(9):
        if r == 3 or r == 6:
            print("------+-------+------")
        row_text = ""
        for c in range(9):
            if c == 3 or c == 6:
                row_text += "| "
            if grid[r][c] == 0:
                row_text += ". "
            else:
                row_text += str(grid[r][c]) + " "
        print(row_text)

In [5]:
# Why: to prove augmentation never breaks a puzzle
def is_valid_solution(cells):
    grid = cells.reshape(9, 9)
    correct = {1, 2, 3, 4, 5, 6, 7, 8, 9}

    for r in range(9):
        if set(grid[r, :].tolist()) != correct:
            return False

    for c in range(9):
        if set(grid[:, c].tolist()) != correct:
            return False

    for box_r in [0, 3, 6]:
        for box_c in [0, 3, 6]:
            box = grid[box_r:box_r+3, box_c:box_c+3]
            if set(box.flatten().tolist()) != correct:
                return False

    return True


def solution_matches_puzzle(puzzle, solution):
    for i in range(81):
        if puzzle[i] != 0 and puzzle[i] != solution[i]:
            return False
    return True

In [6]:
# Why: digits are just symbols, so the logic stays the same
rng = np.random.default_rng(42)

def swap_digits(puzzle, solution):
    new_digits = rng.permutation(9) + 1
    mapping = [0] + list(new_digits)

    new_puzzle = np.array([mapping[x] for x in puzzle])
    new_solution = np.array([mapping[x] for x in solution])
    return new_puzzle, new_solution

In [7]:
# Why: rows stay in their band, so every box keeps its 9 digits
def swap_rows(puzzle, solution):
    p = puzzle.reshape(9, 9)
    s = solution.reshape(9, 9)

    new_order = []
    for band in [0, 3, 6]:
        rows = [band, band + 1, band + 2]
        rng.shuffle(rows)
        new_order = new_order + rows

    return p[new_order].flatten(), s[new_order].flatten()

In [8]:
# Why: a band moves together with its boxes
def swap_bands(puzzle, solution):
    p = puzzle.reshape(9, 9)
    s = solution.reshape(9, 9)

    bands = [0, 3, 6]
    rng.shuffle(bands)

    new_order = []
    for band in bands:
        new_order = new_order + [band, band + 1, band + 2]

    return p[new_order].flatten(), s[new_order].flatten()

In [9]:
# Why: rows become columns, but every rule still holds
def rotate(puzzle, solution):
    k = rng.integers(0, 4)
    p = np.rot90(puzzle.reshape(9, 9), k)
    s = np.rot90(solution.reshape(9, 9), k)
    return p.flatten(), s.flatten()

In [10]:
# Why: all four changes together make each copy look very different
def augment_once(puzzle, solution):
    p, s = swap_digits(puzzle, solution)
    p, s = swap_rows(p, s)
    p, s = swap_bands(p, s)
    p, s = rotate(p, s)
    return p, s

def make_copies(puzzle, solution, n):
    new_puzzles = []
    new_solutions = []
    for i in range(n):
        p, s = augment_once(puzzle, solution)
        new_puzzles.append(p)
        new_solutions.append(s)
    return np.array(new_puzzles), np.array(new_solutions)

In [11]:
copies_p, copies_s = make_copies(puzzles[0], solutions[0], 3)

print("ORIGINAL")
show_grid(puzzles[1])
print()
print("COPY 1")
show_grid(copies_p[1])

ORIGINAL
. . 6 | . . . | . 3 . 
. . 2 | . 4 1 | 9 . 8 
. . 5 | . 7 3 | 6 . . 
------+-------+------
. . . | . . . | . . . 
7 . 3 | . . 9 | 5 2 . 
. 5 . | . . . | . 4 1 
------+-------+------
2 . 9 | . . . | . . . 
. . . | . 8 . | . . . 
. 3 . | . . 5 | . . 9 

COPY 1
3 6 . | . 4 . | . . . 
5 . . | 1 . 2 | 9 . . 
. . . | . . . | . 7 . 
------+-------+------
. . . | 9 . . | . . . 
. . . | . . 1 | 4 . . 
. 3 8 | . . . | . . . 
------+-------+------
. 9 . | 6 . 4 | 7 . 5 
. 8 . | . . . | . . 2 
. 4 . | . 5 . | . . 8 


In [12]:
# Why: a broken copy would teach wrong rules; same clues = same difficulty
count = 0
for i in range(1000):
    copies_p, copies_s = make_copies(puzzles[i], solutions[i], 5)
    for j in range(5):
        valid = is_valid_solution(copies_s[j])
        matches = solution_matches_puzzle(copies_p[j], copies_s[j])
        same_clues = (copies_p[j] != 0).sum() == (puzzles[i] != 0).sum()
        if valid and matches and same_clues:
            count += 1

print("Good copies:", count, "out of 5000")

copies_p, copies_s = make_copies(puzzles[0], solutions[0], 10)
different = len(set(tuple(x) for x in copies_p))
print("Different copies:", different, "out of 10")

Good copies: 5000 out of 5000
Different copies: 10 out of 10


In [13]:
# Why: each original appears k times (1 original + k-1 copies)
# Originals are the first N of the pool, so smaller sets are inside bigger ones
def build_training_set(n_originals, k):
    orig_p = puzzles[:n_originals]
    orig_s = solutions[:n_originals]
    orig_g = groups[:n_originals]

    all_p = []
    all_s = []
    all_g = []
    for i in range(n_originals):
        # keep the original
        all_p.append(orig_p[i])
        all_s.append(orig_s[i])
        all_g.append(orig_g[i])

        # add k-1 augmented copies
        if k > 1:
            copies_p, copies_s = make_copies(orig_p[i], orig_s[i], k - 1)
            for j in range(k - 1):
                all_p.append(copies_p[j])
                all_s.append(copies_s[j])
                all_g.append(orig_g[i])

    return np.array(all_p), np.array(all_s), np.array(all_g)

p, s, g = build_training_set(3, 4)
print(p.shape)
print(g)

(12, 81)
['easy' 'easy' 'easy' 'easy' 'easy' 'easy' 'easy' 'easy' 'hard' 'hard'
 'hard' 'hard']


In [14]:
# Why reset the seed: so the copies are exactly the same every time we run this
rng = np.random.default_rng(42)

settings = [
    (100, 1), (100, 100),
    (1000, 1), (1000, 100),
    (10, 1000), (1000, 10), (10000, 1),
]

for n_originals, k in settings:
    p, s, g = build_training_set(n_originals, k)
    name = "train_n" + str(n_originals) + "_k" + str(k)
    np.savez_compressed(
        "/kaggle/working/" + name + ".npz",
        puzzles=p.astype(np.int8),
        solutions=s.astype(np.int8),
        groups=g,
    )
    print("Saved", name, "-", len(p), "examples")

Saved train_n100_k1 - 100 examples
Saved train_n100_k100 - 10000 examples
Saved train_n1000_k1 - 1000 examples
Saved train_n1000_k100 - 100000 examples
Saved train_n10_k1000 - 10000 examples
Saved train_n1000_k10 - 10000 examples
Saved train_n10000_k1 - 10000 examples


In [15]:
# Why: make sure the saved files are correct before training on them
for n_originals, k in settings:
    name = "train_n" + str(n_originals) + "_k" + str(k)
    d = np.load("/kaggle/working/" + name + ".npz")
    p = d["puzzles"]
    s = d["solutions"]

    check = min(500, len(p))
    good = 0
    for i in range(check):
        if is_valid_solution(s[i]) and solution_matches_puzzle(p[i], s[i]):
            good += 1

    print(name, p.shape, "good:", good, "/", check)

train_n100_k1 (100, 81) good: 100 / 100
train_n100_k100 (10000, 81) good: 500 / 500
train_n1000_k1 (1000, 81) good: 500 / 500
train_n1000_k100 (100000, 81) good: 500 / 500
train_n10_k1000 (10000, 81) good: 500 / 500
train_n1000_k10 (10000, 81) good: 500 / 500
train_n10000_k1 (10000, 81) good: 500 / 500
